
Script générant un attaque adverse simple : rotation. On teste :

- Tous les zooms entre x0.1 et x10 pour toutes les images pour obtenir une accuracy moyenne du réseaux sur un data set en fonction du zoom
- Tous les zomms  entre x0.1 et x10 pour chaque image pour un label donné pour obtenir les zooms maximisant ou minimisant la prediction d'un réseau sur ce label


> ... TODO ... # Adapt the script to zoom transfromation and test networks


In [1]:
from retinotopy import *
welcome()

Running on GPU :  NVIDIA RTX A2000 12GB #GPU= 1
-------------------------------------------------------------------------------------------------
On date 2025-01-05, Running learning on host CONECT-LID-01 with device cuda, pytorch==2.5.1+cu124
-------------------------------------------------------------------------------------------------
Welcome on Linux-6.11.0-13-generic-x86_64-with-glibc2.40


In [2]:
data_set_types

['full', 'bbox']

# testing each network for different ~~~rotations~~~~ (ZOOMS !)


In [2]:
args = Params()
args.size_ratio = 0.1
all_ratios = np.arange(args.size_ratio, (10 + args.size_ratio), 0.1).round(1)
data_set_types = ['raw', 'full', 'bbox']

In [ ]:
im_zero = int((1 - all_ratios[0] ) * 10 )
for data_set_type in data_set_types: 
    print(50*'=')
    data_set_type_test = data_set_type if data_set_type != 'raw' else 'full'
    print(f'{data_set_type=}')    
    args = Params()
    args.do_zoom = True
    args.do_mask = args.do_mask if data_set_type != 'raw' else False
    args.root  = f'{DATAROOT}/Imagenet_{data_set_type_test}' # Directory containing images
    args.folders = ['val'] # type of images to use
    print(50*'-')

    for model_name in  ['resnet18', 'resnet50', 'resnet101'] : #, 
        print(f'{model_name=}')

        for do_polar in [False, True]:
            do_polar = do_polar if data_set_type != 'raw' else False
            args.do_polar = do_polar
            print(f'{args.do_polar=}')
            print(50*'.')
            attack_filename = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_results_attack_zooms.parquet'
            df_filename_acc = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_results_zooms_accuracy.parquet'
            if os.path.isfile(df_filename_acc):
                df_angle = pd.read_parquet(df_filename_acc)
            else:
                print(args)
                df_mean, df_attack = None, None
                print(f'Computing {df_filename_acc}')

                image_dataset = image_datasets_transforms(args)['val']

                model_filename = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '.pt' if data_set_type != 'raw' else None

                print(f"Loading pre-trained resnet {model_filename}")
                model = load_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()
            
                pprint(f'Testing : {model_name}')

                model = model.eval()
                
                with torch.no_grad():
                    acc_zoom =  torch.zeros(len(all_ratios))
                    for i_image, (data, label) in tqdm(enumerate(image_dataset)) : 
                        label_batch = (torch.ones([len(data)])*label).data
                        
                        data = data.to(device)
                
                        outputs = model(data)
                
                        outputs = torch.softmax(outputs, dim=1)
                
                        prior_outputs = outputs[:,label]
                        
                        argmax_out = torch.argmax(prior_outputs[9:]) + 9 
                        argmin_out = torch.argmin(prior_outputs[9:]) + 9

                        argmax_in = torch.argmax(prior_outputs[:10]) 
                        argmin_in = torch.argmin(prior_outputs[:10])
                
                
                        _, preds = torch.max(outputs.data.cpu(), dim=1)
                
                        acc_zoom += ((preds.float() == label_batch)*1.).cpu()

                        
                        df_attack_ = pd.DataFrame({'likelihood_0':[prior_outputs[im_zero].item()], '0_pred':torch.argmax(outputs[im_zero]).item(), 
                                            'likelihood_max_out':prior_outputs[argmax_out].item(), 'max_pos_out':all_ratios[argmax_out.item()], 'max_pred_out':torch.argmax(outputs[argmax_out]).item(), 
                                            'likelihood_min_out':prior_outputs[argmin_out].item(), 'min_pos_out':all_ratios[argmin_out.item()], 'min_pred_out':torch.argmax(outputs[argmin_out]).item(),
                                            'likelihood_max_in':prior_outputs[argmax_in].item(), 'max_pos_in':all_ratios[argmax_in.item()], 'max_pred_in':torch.argmax(outputs[argmax_in]).item(), 
                                            'likelihood_min_in':prior_outputs[argmin_in].item(), 'min_pos_in':all_ratios[argmin_in.item()], 'min_pred_in':torch.argmax(outputs[argmin_in]).item(),
                                            'i_image':i_image, 'filename':image_dataset.imgs[i_image][0], 'label':label})
                        
                        df_attack = store_pandas(df_attack, df_attack_)
                
                    acc_zoom /= len(image_dataset)
                    df_mean_ = pd.DataFrame({'mean_accuracy':[acc_zoom.numpy()], 'var':[all_ratios]})
                    df_mean = store_pandas(df_mean, df_mean_)
                    model.cpu()
                    df_mean.to_parquet(df_filename_acc)
                    df_attack.to_parquet(attack_filename)

In [ ]:
df_attack

In [ ]:
df_zoom

In [ ]:
args.size_ratio = 0.1
all_ratios = np.arange(args.size_ratio, (10 + args.size_ratio), 0.1).round(1)
for model_name in  ['resnet18', 'resnet50', 'resnet101']:    
    print(50*'=')
    print(f'{model_name=}')
    fig, ax = plt.subplots(figsize=(fig_width*phi/3, fig_width/phi/2))
    #for data_set_type, ls in zip(['bbox'], data_set_linestyles):
    for data_set_type, ls in zip(data_set_types, data_set_linestyles):
        print(f'{data_set_type=}')
        print(50*'-')

        for do_polar, color in zip([True, False], ['b', 'r']):
            
            df_filename_acc = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_results_zooms_accuracy.parquet' 

            if os.path.isfile(df_filename_acc):
                df_zoom = pd.read_parquet(df_filename_acc)
                        
                label = 'Retino' if do_polar else 'Cartesian'
                label += f' on {data_set_type}'
                ax.plot(df_zoom['var'][0], df_zoom['mean_accuracy'][0], color=color, ls=ls, lw=1, label=label)
        
    #ax.hlines(xmin=-185, xmax=185, y=1/2, ls='--', ec='gray')
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    #ax.set_xlim(-180, 180)
    ax.set_ylim(0, 1)
    ax.axvline(x=0.1, c='k', ls='--', lw=1)
    display_ratio = np.arange(1, 11, 1.).round(1)
    for zoom in display_ratio:
        ax.axvline(x=zoom, c='k', ls='--', lw=1)
    ax.set_xticks(display_ratio)
    #ax.set_yscale("logit", use_overline=True) #one_half="1/2", 
    #ax.set_yticks([.7, .75, .8])
    ax.set_ylabel('Average Accuracy', font=font)
    ax.set_xlabel('Zoom ratio', font=font)
    plt.legend(bbox_to_anchor=(0.8, 1), loc='best', fontsize=10, edgecolor='none')
    plt.tight_layout()
    # plt.xticks(font=font)
    # plt.yticks(font=font);
    plt.show()

In [ ]:
args.size_ratio = 0.1
fontsize = 20
font = font_manager.FontProperties(weight='normal', size=fontsize)
all_ratios = np.arange(args.size_ratio, (10 + args.size_ratio), 0.1).round(1)
for model_name in  ['resnet101']:    
    print(50*'=')
    print(f'{model_name=}')
    fig, ax = plt.subplots(figsize=(fig_width*phi/3, fig_width/phi/2))
    #for data_set_type, ls in zip(['bbox'], data_set_linestyles):
    for data_set_type, ls in zip(['bbox'], data_set_linestyles):
        print(f'{data_set_type=}')
        print(50*'-')

        for do_polar, color, label_ in zip([False, True], ['royalblue', 'brown'], ['Cartesian coordinates on bounding boxes', 'Retinotpic coordinates on bounding boxes']):

            df_filename_acc = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_results_zooms_accuracy.parquet'
            

            if os.path.isfile(df_filename_acc):
                df_zoom = pd.read_parquet(df_filename_acc)
                        
                label = 'Retino' if do_polar else 'Cartesian'
                label += f' on {data_set_type}'
                ax.plot(df_zoom['var'][0], df_zoom['mean_accuracy'][0], color=color, ls='--', lw=5, label=label_)
        
    #ax.hlines(xmin=-185, xmax=185, y=1/2, ls='--', ec='gray')
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    #ax.set_xlim(-180, 180)
    ax.set_ylim(0, 1)
    #ax.axvline(x=0.1, c='k', ls='--', lw=1)
    display_ratio = np.arange(0, 11, 1.).round(1)
    for zoom in display_ratio:
        ax.axvline(x=zoom, c='k', ls='--', lw=1)
    ax.set_xticks(display_ratio)
    #ax.set_yscale("logit", use_overline=True) #one_half="1/2", 
    #ax.set_yticks([.7, .75, .8])
    ax.set_ylabel('Average Accuracy', font=font)
    ax.set_xlabel('Zoom ratio', font=font)
    plt.legend(bbox_to_anchor=(0.7, .5), loc='center', fontsize=16, edgecolor='none')
    plt.tight_layout()
    # plt.xticks(font=font)
    # plt.yticks(font=font);
    plt.show()

# analysis: average accuracy for different zooms


In [ ]:
acc_plot = {}
for model_name in  ['resnet18', 'resnet50', 'resnet101', ]:
    print(50*'=')
    print(f'{model_name=}')
    for data_set_type, ls in zip(data_set_types, data_set_linestyles):
        print(f'{data_set_type=}')
        print(50*'-')

        for do_polar, color in zip([True, True], ['b', 'r']):

            df_filename_acc = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_results_attack_zooms.parquet'
            

            if os.path.isfile(df_filename_acc):

                df_zoom = pd.read_parquet(df_filename_acc)
                print(do_polar)
                min = accuracy_score(df_zoom['min_pred'], df_zoom["label"])
                print(min, 'min')
                zero = accuracy_score(df_zoom['0_pred'], df_zoom["label"])
                print(zero, 'zero')
                max = accuracy_score(df_zoom['max_pred'], df_zoom["label"])
                print(max, 'max')
                print(zero-min, 'delta1', max-zero, 'delta2')

                acc_plot[get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_min'] = min

                acc_plot[get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_zero'] = zero

In [ ]:
#Rotation attack read out
acc_plot = {}
for model_name in  ['resnet18', 'resnet50', 'resnet101']:
    print(50*'=')
    print(f'{model_name=}')
    for data_set_type, ls in zip(data_set_types, data_set_linestyles):
        print(f'{data_set_type=}')
        print(50*'-')

        for do_polar, color in zip([True, False], ['b', 'r']):

            df_filename_acc = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_results_attack_zooms.parquet'
            

            if os.path.isfile(df_filename_acc):

                df_zoom = pd.read_parquet(df_filename_acc)
                print(do_polar)
                min = accuracy_score(df_zoom['min_pred_in'], df_zoom["label"])
                print(min, 'min_in')
                zero = accuracy_score(df_zoom['0_pred'], df_zoom["label"])
                print(zero, 'zero')
                max = accuracy_score(df_zoom['max_pred_in'], df_zoom["label"])
                print(max, 'max_in')
                print(zero-min, 'delta1', max-zero, 'delta2')

                acc_plot[get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_min_in'] = min

                acc_plot[get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_zero'] = zero


                min = accuracy_score(df_zoom['min_pred_out'], df_zoom["label"])
                print(min, 'min_out')
                zero = accuracy_score(df_zoom['0_pred'], df_zoom["label"])
                print(zero, 'zero')
                max = accuracy_score(df_zoom['max_pred_out'], df_zoom["label"])
                print(max, 'max_out')
                print(zero-min, 'delta1', max-zero, 'delta2')

                acc_plot[get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_min_out'] = min


In [14]:
fig_width = 15
fontsize = 21
font = font_manager.FontProperties(weight='normal', size=fontsize)
dpi = 'figure'
dpi = 200
opts_savefig = dict(dpi=dpi, bbox_inches='tight', pad_inches=0, edgecolor=None)

colors = ['b', 'r', 'k', 'g', 'm', 'y']
fig_width = 20
phi = (np.sqrt(5)+1)/2 # golden ratio for the figures :-)

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
fig, axs = plt.subplots(1, 2, sharex=False, sharey=True, figsize=(fig_width, fig_width/2.61803), width_ratios=[1, 1.8])

titles = ["(A) Zoom out attack", "(B) Zoom invariance"]
model_name = 'resnet101'
#colors_cart = ['lightsteelblue', 'brown']
colors = ['royalblue', 'brown']
acc_full_ret_min = acc_plot[get_filename(data_cache, datetag, 'full', model_name, True) + '_min_out']

acc_full_ret = acc_plot[get_filename(data_cache, datetag, 'full', model_name, True) + '_zero']

acc_full_lin_min = acc_plot[get_filename(data_cache, datetag, 'full', model_name, False) + '_min_out']

acc_full_lin = acc_plot[get_filename(data_cache, datetag, 'full', model_name, False) + '_zero']

acc_bbox_ret_min = acc_plot[get_filename(data_cache, datetag, 'bbox', model_name, True) + '_min_out']

acc_bbox_ret = acc_plot[get_filename(data_cache, datetag, 'bbox', model_name, True) + '_zero']

acc_bbox_lin_min = acc_plot[get_filename(data_cache, datetag, 'bbox', model_name, False) + '_min_out']

acc_bbox_lin = acc_plot[get_filename(data_cache, datetag, 'bbox', model_name, False) + '_zero']

acc_notrain = acc_plot[get_filename(data_cache, datetag, 'raw', model_name, False) + '_zero']

acc_notrain_min = acc_plot[get_filename(data_cache, datetag, 'raw', model_name, False) + '_min_out']


type_mod = ("Cartesian", "Retinotopic")
data_means = {
    'full image': (acc_full_lin, acc_full_ret),
    'bounding box': (acc_bbox_lin, acc_bbox_ret ),

}

x = np.arange(len(type_mod))  # the label locations
width = 0.45  # the width of the bars
multiplier = 0
hatchs = [None, "//"]

for i, (attribute, measurement) in enumerate(data_means.items()):
    offset = width * multiplier
    rects = axs[0].bar(x + offset, measurement, width, label=attribute, alpha =.5, color = colors, hatch=hatchs[i])
    #axs[0].bar_label(rects, padding=2)
    multiplier += 1

data_means = {
    'full image': (acc_full_lin_min, acc_full_ret_min),
    'bounding box': (acc_bbox_lin_min, acc_bbox_ret_min )
}
multiplier = 0

for i, (attribute, measurement) in enumerate(data_means.items()):
    offset = width * multiplier
    rects = axs[0].bar(x + offset, measurement, width, color = colors, hatch=hatchs[i])
    #axs[0].bar_label(rects, padding=2)
    multiplier += 1

axs[0].axhline(y=0.78906, c='gray', ls='dotted', lw=3, label='before attack')
axs[0].axhline(y=0.00012, c='gray', ls='--', lw=3, label='after attack')

axs[0].set_xticks(x + width/2, type_mod)
axs[0].set_ylabel('Accuracy', font=font)
axs[0].legend(loc='upper left', ncols=2, fontsize=15)
axs[0].set_ylim(0, 1)


ax = axs[1]
for data_set_type, ls in zip(['full', 'bbox', 'raw'], data_set_linestyles):
    print(f'{data_set_type=}')
    for model_name in  ['resnet101']:
        print(50*'=')
        print(f'{model_name=}')

        print(50*'-')

        for do_polar, color in zip([False, True],  ['royalblue', 'brown', 'grey']):

            if data_set_type == 'raw' :
                df_filename = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_results_zooms_accuracy.parquet' if not do_polar else 'None'
                color = 'grey'
                lw = 3
                ls = '--'
            else:
                lw = 2
                df_filename = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '_results_zooms_accuracy.parquet'


            if os.path.isfile(df_filename):
                df_zoom = pd.read_parquet(df_filename)
                        
                label = 'Retinotopic coordinates' if do_polar else 'Cartesian coordinates'
                label += f' on {data_set_type} image' if data_set_type != 'raw' else f' without fine tuning'
                ax.plot(df_zoom['var'][0], df_zoom['mean_accuracy'][0], color=color, ls=ls, lw=5, label=label)


ax.tick_params(axis='x', labelsize=fontsize)
ax.tick_params(axis='y', labelsize=fontsize)
#ax.set_xlim(-180, 180)

    
#ax.set_xticks(display_ratio)
ax.set_xscale("log",)
ax.set_xlabel('Zoom ratio', font=font)
display_ratio = np.arange(1, 11, 1).round(1)
for zoom in display_ratio:
    ax.axvline(x=zoom, c='k', ls='--', lw=1)
display_ratio = np.arange(0, 1, .1).round(1)
for zoom in display_ratio:
    ax.axvline(x=zoom, c='k', ls='--', lw=1)
plt.legend(loc='lower center', fontsize=15, edgecolor='none')
plt.tight_layout()
# https://matplotlib.org/stable/gallery/lines_bars_and_markers/bar_label_demo.html
for j, ax in enumerate(axs):
        ax.set_title(titles[j], font=font)
        ax.tick_params(axis='both', labelsize=fontsize)    
        #for i, container in enumerate(ax.containers):
        #    ax.bar_label(container, padding=-15, color='black', fmt='%.3f', rotation=0, label_type = 'edge', fontsize=fontsize, weight='bold')

plt.xticks(font=font)
plt.yticks(font=font)
plt.tight_layout()

In [ ]:
to_save(fig, name='fig-zoom')